# 💻 Notebook do Aluno — Aula 12: Router chains e o conceito de grafo de estado

**Disciplina:** Prompt Engineering and Artificial Intelligence  
**Instituição:** FIAP — Ciência da Computação · 2026  
**Professor:** Jorge Luiz Gomes  
**Aula 12/14 — Módulo 4: LangGraph e Encerramento**  
**⏱️ 1h40min**  
**🔀 Router Chain · StateGraph mental · LangGraph motivação**  
**🔁 Andaime 50%**  

---

## Como usar este notebook

- Rode as células **na ordem**, de cima para baixo (`Shift+Enter`).
- Complete apenas as partes marcadas com `___` e `👉 LACUNA`.
- Não apague o código já pronto — ele é o andaime do lab.
- Salve sua cópia: **Arquivo > Salvar uma cópia no Drive**.

## 📋 Roteiro do Lab

**Lab — Aula 12 · 2º Semestre**  
### Router Chain com 3 rotas para o domínio do grupo ★★★

*Grupo 3–4 · 30 minutos · Google Colab*

1. Complete as 4 lacunas — Literal com destinos reais do domínio, prompt do classificador descrevendo cada rota com exemplos específicos do domínio, persona do handler_conversa, função rotear completa.
2. Teste os 3 cenários — RAG (pergunta sobre documentos do grupo), calculadora (um cálculo relevante para o domínio) e conversa (saudação ou pergunta fora do escopo).
3. Analise o roteamento — troque uma palavra-chave na pergunta que vai para RAG para ver se o classificador ainda acerta. Ex: "documentação" → "manual" → "arquivo" — qual para de rotear corretamente?

> **🎯 Gabarito das lacunas**
>
> L1: Literal["rag", "calculadora", "conversa"] (ou os 3 destinos do domínio)
>
> L2: System prompt descrevendo cada destino com exemplos do domínio real
>
> L3: system="Você é um assistente de [DOMÍNIO]. Responda de forma amigável..."
>
> L4: dados["input"] ; rota.destino ; rotear

---

## 🧩 Notebook Aluno — 50% de lacunas

Complete as lacunas marcadas com `___`.

In [ ]:
!pip install langchain langchain-ollama langchain-chroma pydantic -q

from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda
from pydantic import BaseModel
from typing import Literal
import os
from google.colab import userdata

os.environ["OLLAMA_HOST"]    = "https://ollama.com"
os.environ["OLLAMA_API_KEY"] = userdata.get("OLLAMA_API_KEY")

In [ ]:
llm        = ChatOllama(model="gpt-oss:120b", temperature=0)
embeddings = OllamaEmbeddings(model="nomic-embed-text")
retriever  = Chroma(persist_directory="/content/ckp02",
                    embedding_function=embeddings).as_retriever(search_kwargs={"k":3})

# 👉 LACUNA 1: Rota com Literal — adicionar os destinos do domínio do grupo
class Rota(BaseModel):
    destino: Literal[___, ___, ___]  # ex: "rag", "calculadora", "conversa"

# 👉 LACUNA 2: prompt do classificador — descrever cada destino do domínio
prompt_clf = ChatPromptTemplate.from_messages([
    ("system", """___"""),  # descrever cada rota com exemplos do domínio
    ("human", "{input}"),
])
chain_clf = prompt_clf | llm.with_structured_output(Rota)

# Handler RAG (pronto)
handler_rag = ({"contexto": retriever | RunnableLambda(lambda d: "\n".join(x.page_content for x in d)),
                "input": lambda x: x["input"]}
               | ChatPromptTemplate.from_template("Contexto:\n{contexto}\n\nPergunta: {input}")
               | llm | StrOutputParser())

# 👉 LACUNA 3: implementar handler_conversa (chain simples com system prompt amigável)
handler_conversa = (
    ChatPromptTemplate.from_messages([
        ("system", ___),  # persona do assistente do domínio do grupo
        ("human", "{input}"),
    ]) | llm | StrOutputParser()
)

HANDLERS = {"rag": handler_rag, "calculadora": handler_calculadora, "conversa": handler_conversa}

# 👉 LACUNA 4: implementar a função rotear(dados) e montar router_chain
def rotear(dados: dict) -> str:
    rota = chain_clf.invoke({"input": dados[___]})
    print(f"[ROUTER] {rota.destino}")
    return HANDLERS.get(___, HANDLERS["conversa"]).invoke(dados)

router_chain = RunnableLambda(___)

---

## ✍️ Suas anotações

Registre aqui as observações pedidas no roteiro (qualidade dos resultados, comparações e conclusões do grupo).

## 📚 Referências da aula

- Docs LangChain — RunnableLambda e routing patterns. Como construir chains com branching usando RunnableLambda e with_structured_output. python.langchain.com/docs/how_to/routing
- Docs LangGraph — Conceitos de StateGraph, nodes e edges. A leitura recomendada antes da Aula 13. langchain-ai.github.io/langgraph/concepts/low_level
- Blog Anthropic Engineering — "Building Effective Agents" (2025). Seção sobre orchestrators e subagents — a motivação arquitetural para grafos de estado. anthropic.com/engineering/building-effective-agents
- Livro Russell, S.; Norvig, P. — Inteligência Artificial. 3ª ed. Pearson, 2016. Cap. 3 — Resolução de problemas como busca: a teoria por trás de grafos de estado em IA, base conceitual do LangGraph.
- Livro Polzer, D. — RAG with Python Cookbook. O'Reilly, 2026. O custo de latência de um router baseado em LLM (500ms-2s) e a alternativa de classificador leve sobre embeddings — a fundamentação por trás do classificador de intenção desta aula.
- Livro Gullí, A. — Agentic Design Patterns. O'Reilly, 2025. Cap. 3 — Routing: as três formas de implementar roteamento (regras, classificador de ML, LLM) e o trade-off de custo/latência por trás do Router Chain desta aula.

---

**Próxima Aula — Aula 13** — LangGraph — StateGraph, conditional edges e HITL
  
O diagrama desta aula vira código. Estado TypedDict, add_node(), add_conditional_edges(), MemorySaver, interrupt_before.

---

*Copyright © 2026 Prof. Jorge Luiz Gomes · FIAP · Todos os direitos reservados.*